# Pre-processing

In [ ]:
# import statements

import pandas as pd

In [ ]:
# load train and dev data

path = "/content/drive/MyDrive/DATASCI 266 - NLP/ori_pqal.json"
data = pd.read_json(path).T

FileNotFoundError: File /content/drive/MyDrive/DATASCI 266 - NLP/ori_pqal.json does not exist

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = "/content/drive/My Drive/ori_pqal.json"
data = pd.read_json(path).T

In [ ]:
data.head()

,QUESTION,CONTEXTS,LABELS,MESHES,YEAR,reasoning_required_pred,reasoning_free_pred,final_decision,LONG_ANSWER
21645374,Do mitochondria play a role in remodelling lac...,[Programmed cell death (PCD) is the regulated ...,"[BACKGROUND, RESULTS]","[Alismataceae, Apoptosis, Cell Differentiation...",2011,yes,yes,yes,Results depicted mitochondrial dynamics in viv...
16418930,Landolt C and snellen e acuity: differences in...,[Assessment of visual acuity depends on the op...,"[BACKGROUND, PATIENTS AND METHODS, RESULTS]","[Adolescent, Adult, Aged, Aged, 80 and over, A...",2006,no,no,no,"Using the charts described, there was only a s..."
9488747,"Syncope during bathing in infants, a pediatric...",[Apparent life-threatening events in infants a...,"[BACKGROUND, CASE REPORTS]","[Baths, Histamine, Humans, Infant, Syncope, Ur...",1997,yes,yes,yes,"""Aquagenic maladies"" could be a pediatric form..."
17208539,Are the long-term results of the transanal pul...,[The transanal endorectal pull-through (TERPT)...,"[PURPOSE, METHODS, RESULTS]","[Child, Child, Preschool, Colectomy, Female, H...",2007,yes,no,no,Our long-term study showed significantly bette...
10808977,Can tailored interventions increase mammograph...,[Telephone counseling and tailored print commu...,"[BACKGROUND, DESIGN, PARTICIPANTS, INTERVENTIO...","[Cost-Benefit Analysis, Female, Health Mainten...",2000,yes,no,yes,The effects of the intervention were most pron...


In [ ]:
# Data Pre-processing

# combine all abstracts into one entry
data['CONTEXTS_combined'] = ["".join(entry) for entry in data['CONTEXTS']]

# split data into train
data_shuffled = data.sample(frac=1, random_state=420)

split_index_train = int(0.7 * len(data_shuffled))

split_index_val = int(0.8 * len(data_shuffled))

train = data_shuffled.iloc[:split_index_train]
val = data_shuffled.iloc[split_index_train:split_index_val]
test = data_shuffled.iloc[split_index_val:]


In [ ]:
train.head()

,QUESTION,CONTEXTS,LABELS,MESHES,YEAR,reasoning_required_pred,reasoning_free_pred,final_decision,LONG_ANSWER,CONTEXTS_combined
19854401,Attaining negative margins in breast-conservat...,[The purpose of this survey was to ascertain t...,"[BACKGROUND, STUDY DESIGN, RESULTS]","[Adult, Aged, Breast Neoplasms, Carcinoma, Int...",2009,no,no,no,Results of this survey highlight the wide vari...,The purpose of this survey was to ascertain th...
21481154,Improvements in survival of gynaecological can...,[Our hypothesis is that the adoption of Depart...,"[OBJECTIVE, SETTING, POPULATION, METHODS, MAIN...","[England, Female, Genital Neoplasms, Female, H...",2012,yes,yes,yes,The adoption of the 1999 guidance on gynaecolo...,Our hypothesis is that the adoption of Departm...
15388567,Are sports medicine journals relevant and appl...,[To examine the evidence base of sports medici...,"[OBJECTIVE, METHODS, RESULTS]","[Health Personnel, Humans, Periodicals as Topi...",2004,no,no,no,There is a dearth of studies addressing diagno...,To examine the evidence base of sports medicin...
24298614,Is the 7th TNM edition suitable for biological...,[The clinical and prognostic value of the prev...,"[AIMS, METHODOLOGY, RESULTS]","[Adult, Aged, Female, Gastric Mucosa, Humans, ...",None,yes,yes,yes,Upstaging in EGC based on the revised TNM stag...,The clinical and prognostic value of the previ...
22349635,Do elderly patients benefit from surgery in ad...,[Treatment of elderly cancer patients has gain...,"[BACKGROUND, PATIENTS AND METHODS, RESULTS]","[Aged, Female, Humans, Male, Matched-Pair Anal...",2012,yes,no,no,Elderly patients with MSCC did not benefit fro...,Treatment of elderly cancer patients has gaine...


In [ ]:
train['QUESTION'].iloc[0]

'Attaining negative margins in breast-conservation operations: is there a consensus among breast surgeons?'

In [ ]:
train['CONTEXTS_combined'].iloc[0]

'The purpose of this survey was to ascertain the most common surgical practices for attaining negative (tumor-free) surgical margins in patients desiring breast-conservation treatment for breast cancer to see if a consensus exists for optimal treatment of patients.We sent a survey to 1,000 surgeons interested in the treatment of breast cancer. Three hundred eighty-one surgeons responded to this survey and 351 were used for the analysis (response rate of 38%).Answers showed a large variety in clinical practices among breast surgeons across the country. There was little intraoperative margin analysis; only 48% of surgeons examine the margins grossly with a pathologist and even fewer used frozen sections or imprint cytology. Decisions to reexcise specific margins varied greatly. For example, 57% of surgeons would never reexcise for a positive deep margin, but 53% would always reexcise for a positive anterior margin. Most importantly, there was a large range in answers about acceptable mar

In [ ]:
print(f"""The length of train is {len(train)}.
The length of val is {len(val)}.
The length of test is {len(test)}""")

The length of train is 700.
The length of val is 100.
The length of test is 200


# Sparse Embeddings

In [ ]:
# import statements

!pip install bm25s
import bm25s

In [ ]:
contexts_train = [context.lower() for context in train['CONTEXTS_combined']] # list of documents for context

contexts_train_tokens = bm25s.tokenize(contexts_train, stopwords='en')

context_ids = train.index.to_numpy()

retriever = bm25s.BM25(corpus=context_ids)
retriever.index(contexts_train_tokens)

query = train['QUESTION'].iloc[0]

query_tokens = bm25s.tokenize(query, stopwords = 'en')

results, scores = retriever.retrieve(query_tokens, k=5)

for doc, score in zip(results[0], scores[0]):
    print(f"(Score: {score:.4f}) -> {doc}")

Split strings:   0%|          | 0/700 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/700 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/700 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

(Score: 20.6775) -> 19854401
(Score: 7.6435) -> 19608436
(Score: 6.5405) -> 15065192
(Score: 6.4191) -> 17598882
(Score: 6.0943) -> 15631914


In [ ]:
# compare to proper contexts
train.iloc[0, 1]

['The purpose of this survey was to ascertain the most common surgical practices for attaining negative (tumor-free) surgical margins in patients desiring breast-conservation treatment for breast cancer to see if a consensus exists for optimal treatment of patients.',
 'We sent a survey to 1,000 surgeons interested in the treatment of breast cancer. Three hundred eighty-one surgeons responded to this survey and 351 were used for the analysis (response rate of 38%).',
 'Answers showed a large variety in clinical practices among breast surgeons across the country. There was little intraoperative margin analysis; only 48% of surgeons examine the margins grossly with a pathologist and even fewer used frozen sections or imprint cytology. Decisions to reexcise specific margins varied greatly. For example, 57% of surgeons would never reexcise for a positive deep margin, but 53% would always reexcise for a positive anterior margin. Most importantly, there was a large range in answers about acc

In [ ]:
retriever.retrieve(query_tokens, corpus=context_ids, k=5)[0][0].tolist()

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

[19854401, 19608436, 15065192, 17598882, 15631914]

# Dense Embeddings

In [ ]:
# import statements
!pip install txtai
from txtai import Embeddings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 39.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [txtai]


In [ ]:
context_ids = train.index.astype(str).tolist()
documents = list(zip(context_ids, contexts_train))

In [ ]:
# turn every abstract into numbers that capture its meaning
dense = Embeddings(path="neuml/pubmedbert-base-embeddings")
dense.index(documents)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

In [ ]:
dense_results = dense.search(query, 5)

print("question:", query)
print("correct abstract is at position: 19854401")
print("dense's top 5 results:", dense_results)

question: Attaining negative margins in breast-conservation operations: is there a consensus among breast surgeons?
correct abstract is at position: 19854401
dense's top 5 results: [('19854401', 0.7905054092407227), ('18243752', 0.46343231201171875), ('23177368', 0.45477402210235596), ('15631914', 0.42952385544776917), ('17054994', 0.375396728515625)]


In [ ]:
[pair[0] for pair in dense_results]

['19854401', '18243752', '23177368', '15631914', '17054994']

# RRF

In [ ]:
from collections import defaultdict

def rrf(query, num_docs, dense_embeddings, smoothing=60):
  # retrieve sparse documents
  context_ids = train.index.to_numpy()
  query_tokens = bm25s.tokenize(query, stopwords = 'en')
  sparse_results = retriever.retrieve(query_tokens, corpus=context_ids, k=num_docs)[0][0].tolist()


  # retrieve dense results
  dense_score = dense_embeddings.search(query, num_docs)
  dense_results = [pair[0] for pair in dense_score]

  # calculate rrf
  ranked_lists = [sparse_results, dense_results]
  rrf_scores = defaultdict(float)

  for ranking in ranked_lists:
    for rank, doc_id in enumerate(ranking, start=1):
      clean_id = str(doc_id).strip()
      rrf_scores[clean_id] += 1.0 / (smoothing + rank)

  return sorted(rrf_scores.items(), key=lambda d: d[1], reverse= True)

In [ ]:
train["QUESTION"].iloc[0]

'Attaining negative margins in breast-conservation operations: is there a consensus among breast surgeons?'

In [ ]:
query = train["QUESTION"].iloc[0]

fused_results = rrf(query, 5, dense_embeddings=dense)

for doc_id, score in fused_results:
  print(f"{doc_id}: {score:.5f}")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

19854401: 0.03279
15631914: 0.03101
19608436: 0.01613
18243752: 0.01613
15065192: 0.01587
23177368: 0.01587
17598882: 0.01562
17054994: 0.01538


In [ ]:
fused_list = [int(doc_id) for doc_id, score in fused_results]
train['CONTEXTS_combined'].loc[fused_list]

,CONTEXTS_combined
19854401,The purpose of this survey was to ascertain th...
15631914,Clinically positive axillary nodes are widely ...
19608436,To investigate the contribution of chemical sh...
18243752,This study aims to evaluate local failure patt...
15065192,The use of three-dimensional (3D) ultrasound m...
23177368,Immediate breast reconstruction (IBR) provides...
17598882,A genetic component is well established in the...
17054994,Frozen section (FS) evaluation during thyroid ...


In [ ]:
test.head()

,QUESTION,CONTEXTS,LABELS,MESHES,YEAR,reasoning_required_pred,reasoning_free_pred,final_decision,LONG_ANSWER,CONTEXTS_combined
26104852,Can echocardiography and ECG discriminate here...,[Hereditary transthyretin (ATTR) amyloidosis w...,"[OBJECTIVE, METHODS, RESULTS]","[Adult, Aged, Amino Acid Substitution, Amyloid...",2015,yes,yes,yes,Our study proposes an easily interpretable cla...,Hereditary transthyretin (ATTR) amyloidosis wi...
10759659,The nurse cystoscopist: a feasible option?,[To compare the accuracy achieved by a trained...,"[OBJECTIVES, PATIENTS AND METHODS, RESULTS]","[Cystoscopy, Evaluation Studies as Topic, Feas...",2000,yes,yes,yes,A UNP can be trained to perform cystoscopy and...,To compare the accuracy achieved by a trained ...
26200172,Can biofeedback training of psychophysiologica...,"[In recent years, biofeedback has become incre...","[BACKGROUND, METHODS, RESULTS]","[Adolescent, Adult, Anxiety, Athletic Performa...",2015,yes,yes,yes,One year following completion of the training ...,"In recent years, biofeedback has become increa..."
23348330,Nasal fractures: is closed reduction satisfying?,[This study aims to evaluate the efficacy of c...,"[OBJECTIVE, METHODS, RESULTS]","[Female, Fracture Fixation, Humans, Male, Nose...",2013,maybe,yes,yes,Closed reduction is an easy and sufficient tre...,This study aims to evaluate the efficacy of cl...
17076091,Does obstructive sleep apnea affect aerobic fi...,[We sought to determine whether patients with ...,"[OBJECTIVES, METHODS, RESULTS]","[Adolescent, Adult, Cohort Studies, Continuous...",2006,yes,maybe,maybe,"Overall, in a US Air Force population, OSA doe...",We sought to determine whether patients with o...


## Runnning RRF on all contexts and saving info

In [ ]:
# 1. Full Corpus Sparse Index
all_contexts = [context.lower() for context in data['CONTEXTS_combined']]
all_tokens = bm25s.tokenize(all_contexts, stopwords='en')
all_context_ids = data.index.to_numpy()

full_retriever = bm25s.BM25(corpus=all_context_ids)
full_retriever.index(all_tokens)

# 2. Full Corpus Dense Index
all_str_ids = data.index.astype(str).tolist()
full_dense = Embeddings(path="neuml/pubmedbert-base-embeddings")
full_dense.index(list(zip(all_str_ids, all_contexts)))

Split strings:   0%|          | 0/1000 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/1000 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1000 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
from collections import defaultdict
from tqdm import tqdm
import pandas as pd

def get_rrf_contexts(query, top_k_retrieve=10, top_k_keep=5, smoothing=60):
    # Sparse retrieval
    query_tokens = bm25s.tokenize([query.lower()], stopwords='en')
    sparse_res, _ = full_retriever.retrieve(query_tokens, corpus=all_context_ids, k=top_k_retrieve)
    sparse_ids = [str(doc_id).strip() for doc_id in sparse_res[0]]

    # Dense retrieval
    dense_res = full_dense.search(query, top_k_retrieve)
    dense_ids = [str(pair[0]).strip() for pair in dense_res]

    # Fuse rankings
    rrf_scores = defaultdict(float)
    for ranking in [sparse_ids, dense_ids]:
        for rank, doc_id in enumerate(ranking, start=1):
            rrf_scores[doc_id] += 1.0 / (smoothing + rank)

    # Sort and keep top_k_keep document IDs
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k_keep]
    top_ids = [int(doc_id) for doc_id, _ in sorted_docs]

    # Fetch the actual text strings for those IDs
    retrieved_texts = data.loc[top_ids, 'CONTEXTS_combined'].tolist()
    return top_ids, retrieved_texts

def process_split(df, split_name, top_k_keep=5):
    # If df is a numpy array or list, convert it back to a pandas DataFrame first
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)

    records = []
    for pmid, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        top_ids, retrieved_texts = get_rrf_contexts(row['QUESTION'], top_k_retrieve=10, top_k_keep=top_k_keep)

        records.append({
            "pubid": int(pmid),
            "question": row['QUESTION'],
            "retrieved_ids": top_ids,
            "context": {"contexts": retrieved_texts},  # Matches your format_pubmedqa structure
            "long_answer": row['LONG_ANSWER'],
            "final_decision": row['final_decision']
        })
    return pd.DataFrame(records)

train_rrf = process_split(train, "train", top_k_keep=5)
val_rrf   = process_split(val, "val", top_k_keep=5)
test_rrf  = process_split(test, "test", top_k_keep=5)

Processing train:   0%|          | 0/700 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Processing train:   0%|          | 2/700 [00:00<00:35, 19.55it/s]

... (2,499 outputs left out)

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Processing test: 100%|██████████| 200/200 [00:13<00:00, 14.48it/s]


In [ ]:
save_dir = "/content/drive/My Drive"

train_rrf.to_json(f"{save_dir}/pubmedqa_train_rrf.jsonl", orient="records", lines=True)
val_rrf.to_json(f"{save_dir}/pubmedqa_val_rrf.jsonl", orient="records", lines=True)
test_rrf.to_json(f"{save_dir}/pubmedqa_test_rrf.jsonl", orient="records", lines=True)

# SFT

In [ ]:
import gc
import torch

# Delete your embedding model / previous objects if they exist
# del embed_model

gc.collect()
torch.cuda.empty_cache()

# Verify you have ~14+ GB free before proceeding
free_gb = torch.cuda.mem_get_info()[0] / (1024**3)
print(f"Free VRAM: {free_gb:.2f} GB")

Free VRAM: 21.84 GB


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/drive/My Drive/pubmedqa_train_rrf.jsonl",
        "validation": "/content/drive/My Drive/pubmedqa_val_rrf.jsonl",
        "test": "/content/drive/My Drive/pubmedqa_test_rrf.jsonl"
    }
)
train_ds = dataset["train"]
val_ds = dataset["validation"]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "unsloth/Llama-3.2-1B-Instruct"  # 1B fits comfortably in FP32 on T4
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32, device_map="auto")

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:

def format_pubmedqa(examples):
    texts = []
    for question, context_dict, long_ans, final_decision in zip(
        examples["question"],
        examples["context"],
        examples["long_answer"],
        examples["final_decision"],
    ):
        # Join the retrieved abstracts/contexts
        context_str = "\n".join(context_dict["contexts"])

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a clinical biomedical QA assistant. Synthesize the provided "
                    "literature context carefully, provide a grounded explanation, and conclude "
                    "with a final verdict of 'yes', 'no', or 'maybe'."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context_str}\n\nQuestion: {question}",
            },
            {
                "role": "assistant",
                "content": f"Clinical Synthesis:\n{long_ans}\n\nFinal Decision: {final_decision}",
            },
        ]
        # tokenize=False returns the formatted string with <|eot_id|> tokens appended
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_ds = train_ds.map(format_pubmedqa, batched=True)
val_ds = val_ds.map(format_pubmedqa, batched=True)

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
train_ds

Dataset({
    features: ['pubid', 'question', 'retrieved_ids', 'context', 'long_answer', 'final_decision', 'text'],
    num_rows: 700
})

In [ ]:
train_ds[0]

{'pubid': 19854401,
 'question': 'Attaining negative margins in breast-conservation operations: is there a consensus among breast surgeons?',
 'retrieved_ids': [19854401, 26471488, 18243752, 11034241, 15631914],
 'context': {'contexts': ['The purpose of this survey was to ascertain the most common surgical practices for attaining negative (tumor-free) surgical margins in patients desiring breast-conservation treatment for breast cancer to see if a consensus exists for optimal treatment of patients.We sent a survey to 1,000 surgeons interested in the treatment of breast cancer. Three hundred eighty-one surgeons responded to this survey and 351 were used for the analysis (response rate of 38%).Answers showed a large variety in clinical practices among breast surgeons across the country. There was little intraoperative margin analysis; only 48% of surgeons examine the margins grossly with a pathologist and even fewer used frozen sections or imprint cytology. Decisions to reexcise specific

In [ ]:
!pip install -q trl bitsandbytes

In [ ]:
from trl import SFTTrainer, SFTConfig

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = 2048,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,   # Effective batch size = 8
        warmup_steps = 10,
        num_train_epochs = 1,              # 700 rows / 8 = ~87 steps per epoch
        learning_rate = 2e-5,              # Lower LR is critical for non-LoRA fine-tuning
        fp16 = True,                       # T4 uses FP16
        bf16 = False,
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 20,
        save_strategy = "steps",
        save_steps = 40,
        output_dir = "/content/drive/My Drive/pubmedqa_full_ft",
        optim = "adamw_8bit",              # 8-bit optimizer saves ~75% optimizer VRAM
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

Adding EOS to train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

... (4 outputs left out)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
20,1.868377,1.852457,1.776728,0.588010,259027.000000
40,1.500140,1.547763,1.576941,0.647174,519746.000000
60,1.289719,1.293763,1.493334,0.702012,778100.000000
80,1.190652,1.198700,1.324994,0.721611,1046560.000000
88,1.164911,1.195007,1.304800,0.722161,1144650.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!pip install evaluate

In [ ]:
!pip install rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24985 sha256=15138a45c07937de7daf066349cdb8d77731d2c2aa3817402aef59d3ddeb1446
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score


In [ ]:
import re
import torch
import pandas as pd
from tqdm import tqdm
import evaluate

# Load evaluation metrics
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

model.eval()
test_ds = dataset["test"]

predictions = []
references = []
generated_texts = []

for row in tqdm(test_ds, desc="Generating Test Predictions"):
    context_str = "\n".join(row["context"]["contexts"])

    messages = [
        {
            "role": "system",
            "content": (
                "You are a clinical biomedical QA assistant. Synthesize the provided "
                "literature context carefully, provide a grounded explanation, and conclude "
                "with a final verdict of 'yes', 'no', or 'maybe'."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context_str}\n\nQuestion: {row['question']}",
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,      # Greedy decoding for deterministic evaluation
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Parse out the Final Decision and Clinical Synthesis long answer
    decision_match = re.search(r"Final Decision:\s*(yes|no|maybe)", generated_text, re.IGNORECASE)
    pred_decision = decision_match.group(1).lower() if decision_match else "unknown"

    pred_long_answer = re.sub(r"Clinical Synthesis:\s*", "", generated_text)
    pred_long_answer = re.sub(r"\n*Final Decision:.*", "", pred_long_answer, flags=re.IGNORECASE).strip()

    generated_texts.append(pred_long_answer)
    references.append(row["long_answer"])

    predictions.append({
        "pubid": row["pubid"],
        "question": row["question"],
        "ground_truth_decision": row["final_decision"],
        "predicted_decision": pred_decision,
        "ground_truth_long_answer": row["long_answer"],
        "predicted_long_answer": pred_long_answer,
        "raw_generation": generated_text,
    })

# Compute ROUGE scores
rouge_results = rouge_metric.compute(predictions=generated_texts, references=references)
# Compute METEOR score
meteor_results = meteor_metric.compute(predictions=generated_texts, references=references)

print("\n--- Evaluation Results ---")
print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")
print(f"METEOR:  {meteor_results['meteor']:.4f}")

# Final Decision Classification Accuracy
results_df = pd.DataFrame(predictions)
decision_accuracy = (results_df["ground_truth_decision"] == results_df["predicted_decision"]).mean()
print(f"Final Decision Accuracy: {decision_accuracy:.2%}")

# Save results to Google Drive
results_df.to_json("/content/drive/My Drive/pubmedqa_test_predictions_evaluated.jsonl", orient="records", lines=True)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
Generating Test Predictions:   0%|          | 0/200 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Generat


--- Evaluation Results ---
ROUGE-1: 0.3301
ROUGE-2: 0.1210
ROUGE-L: 0.2464
METEOR:  0.2639
Final Decision Accuracy: 63.50%
